In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

class NeuroFuzzyTSK:
    def __init__(self, num_features=2, learning_rate=0.01):
        self.lr = learning_rate
        self.num_features = num_features
        
        # Initialize Fuzzy Membership Parameters (Gaussian: mean 'c', deviation 'sigma')
        # We will use 2 fuzzy sets (Low, High) per input feature
        self.c = np.array([[0.3, 0.7], [0.3, 0.7]])        # Shape: (features, rules)
        self.sigma = np.array([[0.2, 0.2], [0.2, 0.2]])    # Shape: (features, rules)
        
        # TSK Consequent parameters (Linear function weights for the rules)
        # Rule 1: if x0 is Low and x1 is Low -> y1 = p0*x0 + p1*x1 + r0
        # Rule 2: if x0 is High and x1 is High -> y2 = p2*x0 + p3*x1 + r1
        self.p = np.random.randn(2, num_features)
        self.r = np.random.randn(2)

    def _gaussian_mf(self, x, c, sigma):
        """Gaussian Membership Function"""
        return np.exp(-0.5 * ((x - c) / sigma) ** 2)

    def forward(self, X):
        """Forward pass through the Neuro-Fuzzy layers"""
        num_samples = X.shape[0]
        
        # Layer 1: Fuzzification (Calculate membership degrees)
        w = np.zeros((num_samples, 2))
        for i in range(num_samples):
            # Rule 1: Combination of 'Low' memberships
            mu_low_0 = self._gaussian_mf(X[i, 0], self.c[0, 0], self.sigma[0, 0])
            mu_low_1 = self._gaussian_mf(X[i, 1], self.c[1, 0], self.sigma[1, 0])
            w[i, 0] = mu_low_0 * mu_low_1  # T-norm (product)
            
            # Rule 2: Combination of 'High' memberships
            mu_high_0 = self._gaussian_mf(X[i, 0], self.c[0, 1], self.sigma[0, 1])
            mu_high_1 = self._gaussian_mf(X[i, 1], self.c[1, 1], self.sigma[1, 1])
            w[i, 1] = mu_high_0 * mu_high_1

        # Layer 2: Rule Strength Normalization
        w_sum = np.sum(w, axis=1, keepdims=True) + 1e-8
        w_norm = w / w_sum

        # Layer 3: Consequent / Defuzzification Layer (TSK Model)
        f0 = np.dot(X, self.p[0, :]) + self.r[0]
        f1 = np.dot(X, self.p[1, :]) + self.r[1]
        
        # Final crisp output calculation
        y_pred = w_norm[:, 0] * f0 + w_norm[:, 1] * f1
        return y_pred, w_norm, f0, f1

    def train(self, X, y, epochs=200):
        """Neural Training Phase (Backpropagation over Fuzzy Parameters)"""
        print("Training Neuro-Fuzzy System...")
        for epoch in range(epochs):
            y_pred, w_norm, f0, f1 = self.forward(X)
            loss = np.mean((y_pred - y) ** 2)  # MSE Loss
            
            # Backpropagation / Gradient Descent to update Consequent Weights
            for i in range(X.shape[0]):
                error = y_pred[i] - y[i]
                
                # Update TSK linear rules parameters
                self.p[0, :] -= self.lr * error * w_norm[i, 0] * X[i, :]
                self.r[0]    -= self.lr * error * w_norm[i, 0]
                
                self.p[1, :] -= self.lr * error * w_norm[i, 1] * X[i, :]
                self.r[1]    -= self.lr * error * w_norm[i, 1]
                
                # Dynamic adjustment of Fuzzy Centers (c) to fit data boundaries
                for f in range(self.num_features):
                    # Gradient tuning for Rule 0 Center
                    self.c[f, 0] -= self.lr * error * (f0[i] - y_pred[i]) * w_norm[i, 0] * ((X[i, f] - self.c[f, 0]) / (self.sigma[f, 0]**2))
                    # Gradient tuning for Rule 1 Center
                    self.c[f, 1] -= self.lr * error * (f1[i] - y_pred[i]) * w_norm[i, 1] * ((X[i, f] - self.c[f, 1]) / (self.sigma[f, 1]**2))

            if epoch % 40 == 0 or epoch == epochs - 1:
                print(f"Epoch {epoch:3d} | Mean Squared Error (Loss): {loss:.5f}")

In [2]:
# --- Data Preparation Pipelines ---
iris = load_iris()
# Extracting Versicolor and Virginica classes (Target 1 & 2), using Petal Length & Width
X = iris.data[50:, 2:4] 
y = iris.target[50:] - 1  # Normalize targets to 0 and 1

# Normalize feature scales between 0 and 1 for optimal fuzzy membership profiling
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

# --- Initialization and Execution ---
nf_model = NeuroFuzzyTSK(num_features=2, learning_rate=0.05)
nf_model.train(X_train, y_train, epochs=200)

# Evaluation Pass
test_predictions, _, _, _ = nf_model.forward(X_test)
binary_predictions = np.where(test_predictions > 0.5, 1, 0)
accuracy = np.mean(binary_predictions == y_test) * 100

print("\n--- Model Evaluation Results ---")
print(f"Target Classification Test Accuracy: {accuracy:.2f}%")
print(f"Optimized Fuzzy Centers for Feature 0 (Petal Length): {nf_model.c[0]}")
print(f"Optimized Fuzzy Centers for Feature 1 (Petal Width):  {nf_model.c[1]}")

Training Neuro-Fuzzy System...
Epoch   0 | Mean Squared Error (Loss): 4.12502
Epoch  40 | Mean Squared Error (Loss): 0.52857
Epoch  80 | Mean Squared Error (Loss): 0.52857
Epoch 120 | Mean Squared Error (Loss): 0.52857
Epoch 160 | Mean Squared Error (Loss): 0.52857
Epoch 199 | Mean Squared Error (Loss): 0.52857

--- Model Evaluation Results ---
Target Classification Test Accuracy: 56.67%
Optimized Fuzzy Centers for Feature 0 (Petal Length): [6.18934134e+97 1.72614935e+03]
Optimized Fuzzy Centers for Feature 1 (Petal Width):  [ 9.32051136e+97 -3.30599372e+03]
